In [14]:
import pandas as pd
import numpy as np
import pickle
import os

**For LLM prior**

In [2]:
with open(f'.././replogle_k562_essential_1000hvg_kernels/knowledge_kernels_1k/scgpt_blood/pert_list.pkl','rb') as f:
        gene=pickle.load(f)

In [4]:
#model_name=['gpt-4.1-mini','qwen2.5-7b-instruct','deepseek-r1']
model_name='gpt-4.1-mini'
df=pd.read_csv(f'.././/replogle_k562_essential_1000hvg_kernels/{model_name}_K562_gene_priors.csv')
df=df[['gene_name','hubness_score','phenotype_impact_score','knowledge_scarcity_score','V_llm_raw']]
llm_mean=df['V_llm_raw'].mean()
llm_std=df['V_llm_raw'].std()

df['z_score']=(df['V_llm_raw']-llm_mean)/(llm_std+1e-8)
score1=np.array(df['z_score'])

model_name=f'{model_name}_zcore'
path=f'./knowledge_kernels_1k/{model_name.lower()}'

os.makedirs(path,exist_ok=True)

with open(f'{path}/kernel.pkl','wb') as f:
        pickle.dump(score1,f)

model_name=f'{model_name}_minmaxcore'
path=f'./knowledge_kernels_1k/{model_name.lower()}'

os.makedirs(path,exist_ok=True)

**For other feature kernels construction, we followed the approach described by [Huang et al. (2024)](https://www.biorxiv.org/content/10.1101/2023.12.12.571389v1).  
The detailed code can be found [here](https://github.com/Genentech/iterative-perturb-seq/tree/master). We provide the preprocessing of the scGPT embeddings kernel as an example.**



In [7]:
import pickle
with open('.././replogle_k562_essential_1000hvg_kernels/knowledge_kernels_1k/ground_truth_delta/kernel.pkl','rb') as f:
        kenerl=pickle.load(f)

with open('.././replogle_k562_essential_1000hvg_kernels/knowledge_kernels_1k/ground_truth_delta/pert_list.pkl','rb') as f:
        pert_list=pickle.load(f)

with open(f'.././replogle_k562_essential_1000hvg_kernels/knowledge_kernels_1k/ground_truth_delta/feat.pkl','rb') as f:
        feature=pickle.load(f)

In [5]:
import pandas as pd
import numpy as np
model_name='scGPT_blood'
df=pd.read_csv(f'.././replogle_k562_essential_1000hvg_kernels/replogle_k562_essential_gene_list_2042.csv')
scgpt=pd.read_csv(f'.././replogle_k562_essential_1000hvg_kernels/scgpt_emb/{model_name}_scgpt_gene_emb.csv')
scgpt_gene=scgpt['0']
overlap_gene = pd.merge(df, scgpt_gene, on='0', how='inner')
overlap_gene_list = overlap_gene['0'].tolist()

In [8]:
original_genes = pd.Index(pert_list)
indices_to_keep = original_genes.get_indexer(overlap_gene_list)
valid_indices = indices_to_keep[indices_to_keep >= 0]
cropped_kernel = kenerl[valid_indices, :][:, valid_indices]
cropped_pert_list=overlap_gene_list
cropped_feature=feature[valid_indices, :]

In [11]:
newscgpt=scgpt.iloc[:,2:]
# The dimension of the scGPT embedding is 512, so we assign column names from 0 to 511
newscgpt.columns=list(range(512))
newscgpt=np.array(newscgpt)

In [12]:
G = np.dot(newscgpt, newscgpt.T) 
G

array([[592.40589481,  40.74212149,  -3.52022984, ...,  53.23597193,
         -1.49487625,   4.97454561],
       [ 40.74212149, 589.53990537, -39.19532845, ...,  38.484699  ,
         66.76429016,  -8.31890786],
       [ -3.52022984, -39.19532845, 596.28763699, ...,  -1.74993582,
         13.98673378,   9.42542388],
       ...,
       [ 53.23597193,  38.484699  ,  -1.74993582, ..., 582.63419339,
         26.49973054, -43.29854743],
       [ -1.49487625,  66.76429016,  13.98673378, ...,  26.49973054,
        603.96956304,  18.98817559],
       [  4.97454561,  -8.31890786,   9.42542388, ..., -43.29854743,
         18.98817559, 615.3735147 ]])

In [15]:
path=f'.././replogle_k562_essential_1000hvg_kernels/knowledge_kernels_1k/{model_name.lower()}'

os.makedirs(path,exist_ok=True)

with open(f'{path}/kernel.pkl','wb') as f:
        pickle.dump(G,f)
with open(f'{path}/feat.pkl','wb') as f:
        pickle.dump(newscgpt,f)
with open(f'{path}/pert_list.pkl','wb') as f:
        pickle.dump(cropped_pert_list,f)